# 경찰청 보이스피싱 가설 분석
## 0. 데이터 준비

경찰청 보이스피싱 통계를 활용하여 장기 피해 추세, 사기유형별 피해 차이, 연령대별 피해 분포와 지역별 피해 규모를 확인합니다. 경찰청 데이터 4개는 서로 다른 집계 단위이므로 결합하지 않습니다.

이 Notebook은 기존 `가설검정.ipynb`에서 경찰청 관련 셀을 분리한 독립 실행용 Notebook입니다.

## 0-1. 환경 준비

Google Drive를 연결하고 데이터 처리에 필요한 `pandas`, 경로 처리를 위한 `Path`만 불러옵니다. 그래프를 만들지 않으므로 폰트 설치 코드는 포함하지 않습니다.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path

import pandas as pd
from IPython.display import display

pd.set_option('display.max_columns', 50)
pd.set_option('display.max_colwidth', 100)

## 0-2. 데이터 경로 설정

프로젝트 루트는 `BASE_DIR` 한 곳에서만 관리합니다. Google Drive에 올린 폴더명이 `이종열`과 다르면 아래 한 줄만 수정하세요.

In [ ]:
BASE_DIR = Path('/content/drive/MyDrive/이종열')
DATA_DIR = BASE_DIR / '데이터'
HYPOTHESIS_DIR = BASE_DIR / '가설'
REFERENCE_DIR = BASE_DIR / '분석기준'

file_paths = {
    '경찰청 보이스피싱 현황': DATA_DIR / '경찰청_보이스피싱 현황_20251231.csv',
    '경찰청 피해자 연령별 현황': DATA_DIR / '경찰청_전화금융사기 피해자 연령별 현황_20251231.csv',
    '경찰청 지역별 발생 현황': DATA_DIR / '경찰청_전화금융사기 시도경찰청별 피해 현황_20251231.csv',
    '경찰청 지역별 피해금액': DATA_DIR / '경찰청_전화금융사기_보이스피싱 시도청별 피해금액 현황_20251231.csv',
}

print('프로젝트 루트:', BASE_DIR)
for name, path in file_paths.items():
    print(f'{name}: {path.name} / 존재={path.exists()}')

missing_files = [str(path) for path in file_paths.values() if not path.exists()]
if missing_files:
    raise FileNotFoundError('다음 파일을 찾을 수 없습니다:\n' + '\n'.join(missing_files))

## 0-3. CSV 불러오기

실제 파일 확인 결과 경찰청 원본 4개는 `cp949`로 읽습니다. 각 파일의 인코딩을 명시하고, 읽기에 실패하면 오류가 그대로 드러나도록 합니다.

In [ ]:
df_police_status_raw = pd.read_csv(file_paths['경찰청 보이스피싱 현황'], encoding='cp949')
df_police_age_raw = pd.read_csv(file_paths['경찰청 피해자 연령별 현황'], encoding='cp949')
df_police_region_count_raw = pd.read_csv(file_paths['경찰청 지역별 발생 현황'], encoding='cp949')
df_police_region_amount_raw = pd.read_csv(file_paths['경찰청 지역별 피해금액'], encoding='cp949')

raw_dataframes = {
    '경찰청 보이스피싱 현황': df_police_status_raw,
    '경찰청 피해자 연령별 현황': df_police_age_raw,
    '경찰청 지역별 발생 현황': df_police_region_count_raw,
    '경찰청 지역별 피해금액': df_police_region_amount_raw,
}

for name, df in raw_dataframes.items():
    print(f'{name}: {df.shape}')

### 각 경찰청 데이터의 역할

- **보이스피싱 현황**: 2016~2025년 기관사칭형·대출사기형의 발생건수와 피해액 확인
- **피해자 연령별 현황**: 연도별 연령대 피해자 수 확인
- **지역별 발생 현황**: 18개 시도청의 2016~2025년 발생건수 확인
- **지역별 피해금액**: 18개 시도청의 2023~2025년 피해금액 확인

## 0-4. 원본 데이터 기본 구조 확인

값을 변경하기 전에 `head`, `shape`, 컬럼명, `info`, 기초 통계를 확인합니다. 반복 출력만 줄이기 위해 작은 확인 함수를 사용합니다.

In [ ]:
def show_basic_structure(name, df):
    print('\n' + '=' * 80)
    print(name)
    print('shape:', df.shape)
    print('columns:', df.columns.tolist())
    display(df.head())
    print('\n[info]')
    df.info()
    print('\n[숫자형 기초 통계]')
    display(df.describe())
    print('\n[전체 컬럼 요약]')
    display(df.describe(include='all').transpose())

for name, df in raw_dataframes.items():
    show_basic_structure(name, df)

## 0-5. 결측치 확인

컬럼별 결측 개수와 결측률을 함께 확인합니다. 이 단계에서는 결측 행을 삭제하거나 채우지 않습니다.

In [ ]:
for name, df in raw_dataframes.items():
    missing_report = pd.DataFrame({
        '결측 개수': df.isna().sum(),
        '결측률(%)': (df.isna().mean() * 100).round(2),
    })
    print('\n', name)
    display(missing_report)

## 0-6. 중복 확인

각 경찰청 집계자료에서 완전히 같은 행이 있는지 확인합니다. 중복이 있더라도 집계 단위를 먼저 판단하며 자동 삭제하지 않습니다.

In [ ]:
for name, df in raw_dataframes.items():
    duplicate_count = int(df.duplicated().sum())
    print(f'{name}: 완전중복 추가 행 {duplicate_count}건')
    if duplicate_count > 0:
        display(df[df.duplicated(keep=False)].sort_values(df.columns.tolist()).head(50))

## 0-7. 연도 및 지역 값 확인

경찰청 현황·연령별 자료의 연도 범위와 두 지역 자료의 시도청 범주가 일치하는지 확인합니다.

In [ ]:
print('[경찰청 연도 범위]')
print('보이스피싱 현황:', sorted(df_police_status_raw['구분'].unique()))
print('연령별 현황:', sorted(df_police_age_raw['구분'].unique()))

print('\n[경찰청 지역 범주 일치 여부]')
region_count_set = set(df_police_region_count_raw['시도청'])
region_amount_set = set(df_police_region_amount_raw['시도청'])
print('발생 현황에만 존재:', sorted(region_count_set - region_amount_set))
print('피해금액 현황에만 존재:', sorted(region_amount_set - region_count_set))

## 0-8. dtype 및 값 형식 확인

컬럼명·문자열 앞뒤 공백, 숫자형이어야 할 값의 변환 실패 여부, 연도와 피해금액 단위를 확인합니다. 경찰청 보이스피싱 현황의 피해액 컬럼은 이름과 원문 기준 억원 단위입니다.

In [ ]:
for name, df in raw_dataframes.items():
    column_space_count = sum(column != column.strip() for column in df.columns)
    string_space_rows = {}
    for column in df.select_dtypes(include=['object', 'string']).columns:
        values = df[column].dropna().astype(str)
        count = int(values.ne(values.str.strip()).sum())
        if count > 0:
            string_space_rows[column] = count
    print(f'{name}: 컬럼명 공백={column_space_count}, 문자열 값 공백={string_space_rows}')

In [ ]:
expected_numeric_columns = {
    '경찰청 보이스피싱 현황': df_police_status_raw.columns.tolist(),
    '경찰청 피해자 연령별 현황': df_police_age_raw.columns.tolist(),
    '경찰청 지역별 발생 현황': df_police_region_count_raw.columns[1:].tolist(),
    '경찰청 지역별 피해금액': df_police_region_amount_raw.columns[1:].tolist(),
}

for name, columns in expected_numeric_columns.items():
    df = raw_dataframes[name]
    print(f'\n[{name}]')
    for column in columns:
        converted = pd.to_numeric(df[column], errors='coerce')
        new_missing = int((converted.isna() & df[column].notna()).sum())
        print(f'{column}: 현재 dtype={df[column].dtype}, 숫자 변환 실패={new_missing}건')

In [ ]:
print('[피해금액 단위 확인]')
print('경찰청 보이스피싱 현황 피해액 단위: 컬럼명 기준 억원')
print('경찰청 지역별 피해금액 단위: 원문 문서에서 별도 명시가 없어 사람이 확인 필요')

## 0-9. 최소 전처리

원본 DataFrame을 보호하기 위해 복사본에서만 작업합니다. 컬럼명과 문자열의 앞뒤 공백을 안전하게 제거하고, 숫자형이어야 하는 컬럼만 명시적으로 변환합니다. 의미가 다른 범주를 임의로 합치거나 행을 삭제하지 않습니다.

In [ ]:
df_police_status = df_police_status_raw.copy()
df_police_age = df_police_age_raw.copy()
df_police_region_count = df_police_region_count_raw.copy()
df_police_region_amount = df_police_region_amount_raw.copy()

prepared_dataframes = {
    '경찰청 보이스피싱 현황': df_police_status,
    '경찰청 피해자 연령별 현황': df_police_age,
    '경찰청 지역별 발생 현황': df_police_region_count,
    '경찰청 지역별 피해금액': df_police_region_amount,
}

for df in prepared_dataframes.values():
    df.columns = df.columns.str.strip()
    for column in df.select_dtypes(include=['object', 'string']).columns:
        df[column] = df[column].str.strip()

In [ ]:
# 경찰청: 연도를 뜻하는 '구분'을 분석 목적이 분명한 '연도'로 변경합니다.
df_police_status = df_police_status.rename(columns={'구분': '연도'})
df_police_age = df_police_age.rename(columns={'구분': '연도'})

police_status_numeric = df_police_status.columns.tolist()
police_age_numeric = df_police_age.columns.tolist()
police_region_count_numeric = df_police_region_count.columns[1:].tolist()
police_region_amount_numeric = df_police_region_amount.columns[1:].tolist()

for column in police_status_numeric:
    df_police_status[column] = pd.to_numeric(df_police_status[column], errors='coerce')
for column in police_age_numeric:
    df_police_age[column] = pd.to_numeric(df_police_age[column], errors='coerce')
for column in police_region_count_numeric:
    df_police_region_count[column] = pd.to_numeric(df_police_region_count[column], errors='coerce')
for column in police_region_amount_numeric:
    df_police_region_amount[column] = pd.to_numeric(df_police_region_amount[column], errors='coerce')

print('숫자형 변환 후 새 결측 확인')
for name, df in {
    '경찰청 보이스피싱 현황': df_police_status,
    '경찰청 피해자 연령별 현황': df_police_age,
    '경찰청 지역별 발생 현황': df_police_region_count,
    '경찰청 지역별 피해금액': df_police_region_amount,
}.items():
    print(name, int(df.isna().sum().sum()))

## 0-10. 필요한 파생변수 확인 및 생성

최신 경찰청 가설 문서의 P1·P2에 필요한 `전체_발생건수`, `전체_피해액_억원`만 생성합니다. 지역별 건당 피해금액은 분석 단계의 계산이므로 아직 만들지 않습니다.

In [ ]:
if '전체_발생건수' not in df_police_status.columns:
    df_police_status['전체_발생건수'] = (
        df_police_status['기관사칭형_발생건수']
        + df_police_status['대출사기형_발생건수']
    )

if '전체_피해액_억원' not in df_police_status.columns:
    df_police_status['전체_피해액_억원'] = (
        df_police_status['기관사칭형_피해액_억원']
        + df_police_status['대출사기형_피해액_억원']
    )

display(df_police_status[['연도', '전체_발생건수', '전체_피해액_억원']])

## 0-11. 전처리 결과 최종 확인

전처리 전후 shape, 결측, 완전중복, dtype과 생성 변수를 비교합니다.

In [ ]:
final_dataframes = {
    '경찰청 보이스피싱 현황': df_police_status,
    '경찰청 피해자 연령별 현황': df_police_age,
    '경찰청 지역별 발생 현황': df_police_region_count,
    '경찰청 지역별 피해금액': df_police_region_amount,
}

comparison_rows = []
raw_for_comparison = {
    '경찰청 보이스피싱 현황': df_police_status_raw,
    '경찰청 피해자 연령별 현황': df_police_age_raw,
    '경찰청 지역별 발생 현황': df_police_region_count_raw,
    '경찰청 지역별 피해금액': df_police_region_amount_raw,
}

for name, final_df in final_dataframes.items():
    raw_df = raw_for_comparison[name]
    comparison_rows.append({
        '데이터': name,
        '전처리 전 shape': str(raw_df.shape),
        '전처리 후 shape': str(final_df.shape),
        '최종 결측': int(final_df.isna().sum().sum()),
        '최종 완전중복 추가 행': int(final_df.duplicated().sum()),
        '원본에서 삭제한 행': 0,
    })

display(pd.DataFrame(comparison_rows))

In [ ]:
print('[최종 dtype]')
for name, df in final_dataframes.items():
    print(f'\n{name}')
    print(df.dtypes.to_string())

print('\n[최종 생성 변수]')
print('경찰청 보이스피싱 현황: 전체_발생건수, 전체_피해액_억원')
print('범주 통일: 실제 표기 차이가 확인되지 않아 적용하지 않음')
print('이상치 삭제: 0건')
print('원본 CSV 저장/덮어쓰기: 수행하지 않음')

## 0-12. 0단계 요약

아래 셀은 실행된 경찰청 DataFrame에서 실제 숫자를 계산해 요약합니다. 지역별 피해금액 단위는 사람이 확인해야 합니다.

In [ ]:
summary_items = [
    ('경찰청 보이스피싱 현황', df_police_status, '전체_발생건수, 전체_피해액_억원 생성'),
    ('경찰청 연령별 현황', df_police_age, '구분을 연도로 명확화'),
    ('경찰청 지역별 발생 현황', df_police_region_count, '숫자형 확인 및 문자열 공백 정리'),
    ('경찰청 지역별 피해금액', df_police_region_amount, '숫자형 확인 및 문자열 공백 정리'),
]

print('=' * 60)
print('0단계 경찰청 데이터 준비 결과')
print('=' * 60)
for name, df, treatment in summary_items:
    print(f'\n[{name}]')
    print('shape:', df.shape)
    print('결측:', int(df.isna().sum().sum()), '개')
    print('완전중복 추가 행:', int(df.duplicated().sum()), '건')
    print('처리한 내용:', treatment)
    print('삭제한 행: 0건')

print('\n[사람이 확인할 사항]')
print('1. 경찰청 지역별 피해금액 CSV의 금액 단위')

print('=' * 60)
print('0단계 완료')
print('다음 단계: 1. 기본 EDA')
print('=' * 60)

## **0단계 경찰청 데이터 준비 완료**

현재 Notebook에서는 경찰청 4개 데이터의 로드, 구조 확인, 결측·중복·dtype 확인 및 최소 전처리까지 수행했습니다.

# 1. 경찰청 기본 EDA

0단계에서 준비한 경찰청 DataFrame을 그대로 사용해 연도·사기유형·연령대·지역별 기본 현황을 확인합니다. 건수, 비율과 기본 시각화만 기술하며 인과관계를 판단하지 않습니다.

## 1-1. 시각화 환경과 한글 폰트 설정

Google Colab에 나눔고딕을 한 번만 설치한 뒤 현재 런타임의 Matplotlib에 직접 등록합니다. 이 방식은 일반적으로 런타임 재시작이 필요 없습니다. 설치 셀 실행 후에도 한글이 깨지면 셀을 한 번 더 실행하세요.

In [ ]:
!apt-get update -qq
!apt-get install -qq fonts-nanum

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
from matplotlib.ticker import FuncFormatter

font_path = '/usr/share/fonts/truetype/nanum/NanumGothic.ttf'
fm.fontManager.addfont(font_path)
plt.rcParams['font.family'] = 'NanumGothic'
plt.rcParams['axes.unicode_minus'] = False

print('Matplotlib 한글 폰트:', plt.rcParams['font.family'])

## 1-2. 경찰청 보이스피싱 연도별 현황

`df_police_status`에서 연도별 전체·유형별 발생건수와 피해액, 유형별 구성비를 확인합니다. 0단계에서 만든 `전체_발생건수`, `전체_피해액_억원`을 그대로 사용합니다.

### P1. 연도별 보이스피싱 발생건수 변화

**연구 질문:** 2016~2025년 동안 보이스피싱 발생건수는 어떻게 변화했는가?

**사용 변수:** 연도, 기관사칭형_발생건수, 대출사기형_발생건수, 전체_발생건수

**분석 목적:** 보이스피싱이 지속적으로 발생하는 금융 피해인지 기본 추세를 확인한다.

In [ ]:
police_count_table = df_police_status[[
    '연도', '기관사칭형_발생건수', '대출사기형_발생건수', '전체_발생건수'
]].copy()
police_count_table['기관사칭형_비율(%)'] = (
    police_count_table['기관사칭형_발생건수'] / police_count_table['전체_발생건수'] * 100
).round(2)
police_count_table['대출사기형_비율(%)'] = (
    police_count_table['대출사기형_발생건수'] / police_count_table['전체_발생건수'] * 100
).round(2)

display(police_count_table)

In [ ]:
display(police_count_table[['연도', '전체_발생건수']])

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(police_count_table['연도'], police_count_table['전체_발생건수'], marker='o')
ax.set_title('연도별 전체 보이스피싱 발생건수')
ax.set_xlabel('연도')
ax.set_ylabel('발생건수(건)')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

### P3. 사기유형별 피해 규모 차이 - 발생건수

기관사칭형과 대출사기형의 연도별 발생건수를 같은 축에서 비교합니다. 이는 기본 현황 비교이며 통계적 유의성을 판단하지 않습니다.

In [ ]:
display(police_count_table[['연도', '기관사칭형_발생건수', '대출사기형_발생건수']])

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(police_count_table['연도'], police_count_table['기관사칭형_발생건수'], marker='o', label='기관사칭형')
ax.plot(police_count_table['연도'], police_count_table['대출사기형_발생건수'], marker='o', label='대출사기형')
ax.set_title('연도별 사기유형 발생건수')
ax.set_xlabel('연도')
ax.set_ylabel('발생건수(건)')
ax.legend()
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

### P2. 연도별 보이스피싱 피해금액 변화

**연구 질문:** 2016~2025년 동안 전체 피해금액은 어떻게 변화했는가?

**사용 변수:** 연도, 기관사칭형_피해액_억원, 대출사기형_피해액_억원, 전체_피해액_억원

**분석 목적:** 실제 금전적 피해 규모와 연도별 변화를 확인한다.

In [ ]:
police_amount_table = df_police_status[[
    '연도', '기관사칭형_피해액_억원', '대출사기형_피해액_억원', '전체_피해액_억원'
]].copy()
police_amount_table['기관사칭형_피해액_비율(%)'] = (
    police_amount_table['기관사칭형_피해액_억원'] / police_amount_table['전체_피해액_억원'] * 100
).round(2)
police_amount_table['대출사기형_피해액_비율(%)'] = (
    police_amount_table['대출사기형_피해액_억원'] / police_amount_table['전체_피해액_억원'] * 100
).round(2)

display(police_amount_table)

In [ ]:
display(police_amount_table[['연도', '전체_피해액_억원']])

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(police_amount_table['연도'], police_amount_table['전체_피해액_억원'], marker='o')
ax.set_title('연도별 전체 보이스피싱 피해금액')
ax.set_xlabel('연도')
ax.set_ylabel('피해금액(억원)')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

### P3. 사기유형별 피해 규모 차이 - 피해금액

기관사칭형과 대출사기형의 연도별 피해금액을 비교합니다. 피해금액은 원본의 억원 단위를 유지합니다.

In [ ]:
display(police_amount_table[['연도', '기관사칭형_피해액_억원', '대출사기형_피해액_억원']])

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(police_amount_table['연도'], police_amount_table['기관사칭형_피해액_억원'], marker='o', label='기관사칭형')
ax.plot(police_amount_table['연도'], police_amount_table['대출사기형_피해액_억원'], marker='o', label='대출사기형')
ax.set_title('연도별 사기유형 피해금액')
ax.set_xlabel('연도')
ax.set_ylabel('피해금액(억원)')
ax.legend()
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

## 1-3. 경찰청 피해자의 연령대 구성

전체 기간의 연령대별 피해자 수와 비율, 각 연도의 구성비, 2016년과 2025년 구성을 확인합니다.

> 이 데이터는 연령대별 전체 인구를 분모로 한 피해율이 아니라 실제 피해자 구성입니다. 따라서 특정 연령대가 보이스피싱에 당할 확률이 높다고 단정하지 않습니다.

### P4. 연령대별 피해자 분포

**연구 질문:** 전체 피해자 구성은 어떤 연령대에 많이 분포하는가?

**사용 변수:** 20대이하, 30대, 40대, 50대, 60대, 70대이상

**분석 목적:** 전체 기간의 연령대별 피해자 수와 구성비를 확인한다.

In [ ]:
police_age_columns = ['20대이하', '30대', '40대', '50대', '60대', '70대이상']

police_age_total_table = pd.DataFrame({
    '피해자수(명)': df_police_age[police_age_columns].sum()
})
police_age_total_table['비율(%)'] = (
    police_age_total_table['피해자수(명)'] / police_age_total_table['피해자수(명)'].sum() * 100
).round(2)

display(police_age_total_table)

In [ ]:
display(police_age_total_table[['피해자수(명)']])

fig, ax = plt.subplots(figsize=(9, 5))
ax.bar(police_age_total_table.index, police_age_total_table['피해자수(명)'])
ax.set_title('2016~2025년 연령대별 피해자 수')
ax.set_xlabel('연령대')
ax.set_ylabel('피해자수(명)')
plt.tight_layout()
plt.show()

In [ ]:
display(police_age_total_table[['비율(%)']])

fig, ax = plt.subplots(figsize=(9, 5))
ax.bar(police_age_total_table.index, police_age_total_table['비율(%)'])
ax.set_title('2016~2025년 연령대별 피해자 구성비')
ax.set_xlabel('연령대')
ax.set_ylabel('비율(%)')
plt.tight_layout()
plt.show()

### P5. 연령대별 피해 구조의 연도 변화

**연구 질문:** 피해자의 연령대 구성은 연도에 따라 어떻게 달라졌는가?

**분석 목적:** 연도별 연령대 구성비와 2016년·2025년의 차이를 기술적으로 확인한다.

In [ ]:
police_age_ratio_table = df_police_age[['연도'] + police_age_columns].copy()
annual_age_total = police_age_ratio_table[police_age_columns].sum(axis=1)
police_age_ratio_table[police_age_columns] = (
    police_age_ratio_table[police_age_columns].div(annual_age_total, axis=0) * 100
).round(2)

police_age_comparison_table = police_age_ratio_table[
    police_age_ratio_table['연도'].isin([2016, 2025])
].set_index('연도').transpose()

print('[연도별 연령대 구성비(%)]')
display(police_age_ratio_table)
print('[2016년과 2025년 구성비 비교(%)]')
display(police_age_comparison_table)

In [ ]:
display(police_age_ratio_table)

fig, ax = plt.subplots(figsize=(11, 6))
for column in police_age_columns:
    ax.plot(police_age_ratio_table['연도'], police_age_ratio_table[column], marker='o', label=column)
ax.set_title('연도별 피해자 연령대 구성비 변화')
ax.set_xlabel('연도')
ax.set_ylabel('비율(%)')
ax.legend(ncol=3)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

## 1-4. 경찰청 지역별 발생건수

18개 시도청별 2016~2025년 전체 발생건수와 최신 연도 발생건수를 확인합니다. 지역별 인구 규모가 다르므로 총 발생건수가 많다는 사실을 해당 지역 거주자의 피해 확률로 해석하지 않습니다.

### P6. 지역별 보이스피싱 발생규모 차이

**연구 질문:** 지역별 전체 및 최근 연도 발생건수는 어떻게 분포하는가?

**분석 목적:** 보이스피싱이 전국적으로 발생하는 금융 문제인지 기본 규모를 확인한다.

In [ ]:
region_count_year_columns = [column for column in df_police_region_count.columns if column.endswith('년')]
latest_count_year = max(region_count_year_columns, key=lambda value: int(value.replace('년', '')))

police_region_count_table = df_police_region_count[['시도청'] + region_count_year_columns].copy()
police_region_count_table['전체_발생건수'] = police_region_count_table[region_count_year_columns].sum(axis=1)
police_region_count_table['전체_비율(%)'] = (
    police_region_count_table['전체_발생건수'] / police_region_count_table['전체_발생건수'].sum() * 100
).round(2)
police_region_count_table['전체_순위'] = (
    police_region_count_table['전체_발생건수'].rank(method='min', ascending=False).astype(int)
)
police_region_count_table = police_region_count_table.sort_values('전체_발생건수', ascending=False)

display(police_region_count_table)

In [ ]:
region_total_plot_table = police_region_count_table.sort_values('전체_발생건수')
display(region_total_plot_table[['시도청', '전체_발생건수', '전체_비율(%)']])

fig, ax = plt.subplots(figsize=(10, 7))
ax.barh(region_total_plot_table['시도청'], region_total_plot_table['전체_발생건수'])
ax.set_title('2016~2025년 지역별 전체 발생건수')
ax.set_xlabel('발생건수(건)')
ax.set_ylabel('시도청')
plt.tight_layout()
plt.show()

In [ ]:
region_latest_count_table = police_region_count_table[
    ['시도청', latest_count_year]
].sort_values(latest_count_year)
display(region_latest_count_table)

fig, ax = plt.subplots(figsize=(10, 7))
ax.barh(region_latest_count_table['시도청'], region_latest_count_table[latest_count_year])
ax.set_title(f'{latest_count_year} 지역별 보이스피싱 발생건수')
ax.set_xlabel('발생건수(건)')
ax.set_ylabel('시도청')
plt.tight_layout()
plt.show()

## 1-5. 경찰청 지역별 피해금액

2023~2025년 지역별 피해금액과 합계를 확인합니다. 원본 CSV와 최신 가설 문서에 지역 피해금액의 단위가 명시되지 않았으므로 그래프에는 **원본 단위(확인 필요)**라고 표시합니다. 단위가 확인되기 전까지 원·만원·억원으로 임의 환산하지 않습니다. 또한 이번 단계에서는 지역별 건당 피해금액을 계산하지 않습니다.

### P7. 지역별 피해금액 차이

**연구 질문:** 2023~2025년 지역별 피해금액 규모는 어떻게 다른가?

**분석 목적:** 지역별 총 피해금액과 최근 연도 피해금액을 확인한다. 원본 단위가 확정되기 전에는 임의 환산하지 않는다.

In [ ]:
region_amount_year_columns = [column for column in df_police_region_amount.columns if column.endswith('년')]
latest_amount_year = max(region_amount_year_columns, key=lambda value: int(value.replace('년', '')))

police_region_amount_table = df_police_region_amount[['시도청'] + region_amount_year_columns].copy()
police_region_amount_table['2023~2025_총피해금액'] = police_region_amount_table[region_amount_year_columns].sum(axis=1)
police_region_amount_table['총피해금액_순위'] = (
    police_region_amount_table['2023~2025_총피해금액'].rank(method='min', ascending=False).astype(int)
)
police_region_amount_table = police_region_amount_table.sort_values('2023~2025_총피해금액', ascending=False)

display(police_region_amount_table)

In [ ]:
region_total_amount_plot_table = police_region_amount_table.sort_values('2023~2025_총피해금액')
display(region_total_amount_plot_table[['시도청', '2023~2025_총피해금액']])

fig, ax = plt.subplots(figsize=(10, 7))
ax.barh(region_total_amount_plot_table['시도청'], region_total_amount_plot_table['2023~2025_총피해금액'])
ax.set_title('2023~2025년 지역별 총 피해금액')
ax.set_xlabel('피해금액(원본 단위, 확인 필요)')
ax.set_ylabel('시도청')
plt.tight_layout()
plt.show()

In [ ]:
region_latest_amount_table = police_region_amount_table[
    ['시도청', latest_amount_year]
].sort_values(latest_amount_year)
display(region_latest_amount_table)

fig, ax = plt.subplots(figsize=(10, 7))
ax.barh(region_latest_amount_table['시도청'], region_latest_amount_table[latest_amount_year])
ax.set_title(f'{latest_amount_year} 지역별 보이스피싱 피해금액')
ax.set_xlabel('피해금액(원본 단위, 확인 필요)')
ax.set_ylabel('시도청')
plt.tight_layout()
plt.show()

### P8. 지역별 건당 피해금액 비교

**연구 질문:** 지역에 따라 보이스피싱 1건당 피해금액에 차이가 있는가?

기존 `가설검정.ipynb`의 현재 범위에는 P8 계산 코드가 아직 구현되어 있지 않습니다. 이번 작업은 Notebook 분리이므로 새로운 계산을 추가하지 않고, 지역별 피해금액 단위 확인 후 후속 분석에서 진행합니다.

# 경찰청 기본 EDA 요약

아래 셀은 실행된 경찰청 집계표에서 실제 값을 계산해 요약합니다. 수치는 하드코딩하지 않으며 관찰된 현황만 기술합니다.

In [ ]:
count_peak_row = police_count_table.loc[police_count_table['전체_발생건수'].idxmax()]
amount_peak_row = police_amount_table.loc[police_amount_table['전체_피해액_억원'].idxmax()]
latest_count_row = police_count_table.sort_values('연도').iloc[-1]
previous_count_row = police_count_table.sort_values('연도').iloc[-2]
recent_count_change = latest_count_row['전체_발생건수'] - previous_count_row['전체_발생건수']
recent_count_direction = '증가' if recent_count_change > 0 else '감소' if recent_count_change < 0 else '동일'
total_impersonation_count = police_count_table['기관사칭형_발생건수'].sum()
total_loan_count = police_count_table['대출사기형_발생건수'].sum()
dominant_count_type = '기관사칭형' if total_impersonation_count > total_loan_count else '대출사기형'
total_impersonation_amount = police_amount_table['기관사칭형_피해액_억원'].sum()
total_loan_amount = police_amount_table['대출사기형_피해액_억원'].sum()
dominant_amount_type = '기관사칭형' if total_impersonation_amount > total_loan_amount else '대출사기형'

top_police_age = police_age_total_table['피해자수(명)'].idxmax()
top_region_count = police_region_count_table.iloc[0]
top_region_amount = police_region_amount_table.iloc[0]

print('=' * 65)
print('경찰청 기본 EDA 요약')
print('=' * 65)
print(f"발생건수가 가장 많았던 연도: {int(count_peak_row['연도'])}년 ({int(count_peak_row['전체_발생건수']):,}건)")
print(f"피해금액이 가장 컸던 연도: {int(amount_peak_row['연도'])}년 ({amount_peak_row['전체_피해액_억원']:,.0f}억원)")
print(f"최근 발생 변화: {int(previous_count_row['연도'])}년 대비 {int(latest_count_row['연도'])}년 {abs(int(recent_count_change)):,}건 {recent_count_direction}")
print(f"전체 기간 유형별 발생건수: 기관사칭형 {int(total_impersonation_count):,}건, 대출사기형 {int(total_loan_count):,}건 ({dominant_count_type}이 더 많음)")
print(f"전체 기간 유형별 피해금액: 기관사칭형 {total_impersonation_amount:,.0f}억원, 대출사기형 {total_loan_amount:,.0f}억원 ({dominant_amount_type}이 더 큼)")
print(f"전체 기간 피해자 수가 가장 많은 연령대: {top_police_age}")
print(f"전체 기간 발생건수가 가장 많은 지역: {top_region_count['시도청']} ({int(top_region_count['전체_발생건수']):,}건)")
print(f"2023~2025 총 피해금액 원본 수치가 가장 큰 지역: {top_region_amount['시도청']} ({top_region_amount['2023~2025_총피해금액']:,.0f}, 단위 확인 필요)")

print('\n[사람이 확인할 사항]')
print('- 경찰청 지역별 피해금액의 원본 단위')
print('- P8 지역별 건당 피해금액은 현재 미구현')

## **경찰청 기본 EDA 완료**

경찰청 4개 데이터의 연도·사기유형·연령대·지역별 기본 현황을 확인했습니다. 지역 피해금액 단위와 P8 계산은 후속 확인이 필요합니다.